# catification Dataset Builder

Single notebook pipeline:
1. Extract candidate frames from videos
1. Select diverse and low-confidence candidates using embeddings
1. Interactively review cats
1. Export ROI crops and labels
1. Train and tune classifiers on embeddings and compare inference cost
    - RBF SVM
    - MLP
    - Gradient Boosted Trees
1. Repeat iteratively to improve dataset quality and model performance

## Configure notebook

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import random
import joblib
from datetime import datetime
from time import perf_counter

import cv2
import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

import processing
import tracking

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Define constants
TARGET_LABEL_FRAMES = 10
TRAIN_SPLIT = 0.8
SOURCE_SPLITS = ["train", "val"]
CAT_NAMES = {
    0: "unknown",
    1: "fluffy",
    2: "tabby",
}
SEARCH_ITERS = 10
INFERENCE_BUDGET_MS = 5
SCALE_FACTOR = 20
BENCHMARK_RUNS = 100

In [ ]:
# Define paths
EXPORT_ROOT = PROJECT_ROOT / "datasets" / "classification_data"
LABEL_MANIFEST_PATH = EXPORT_ROOT / "labels_manifest.csv"
MODEL_EXPORT_PATH = PROJECT_ROOT / "models_staging" / "classification_best_model.joblib"
MODEL_CANDIDATES_DIR = (
    PROJECT_ROOT / "models_staging" / "classification_tuned_candidates"
)
os.makedirs(MODEL_CANDIDATES_DIR, exist_ok=True)

In [ ]:
# Define functions
def image_path_for_split(split: str, image_name: str) -> Path:
    """Build absolute path for an exported ROI image in a split"""
    return EXPORT_ROOT / "images" / split / image_name


def load_label_manifest(path: Path) -> pd.DataFrame:
    """Load labels manifest with columns: image_name, split, cat_name"""
    if path.exists():
        df = pd.read_csv(path)
        required = {"image_name", "split", "cat_name"}
        if not required.issubset(df.columns):
            raise ValueError(f"Manifest missing columns: {required}")
        return df.copy()
    return pd.DataFrame(columns=["image_name", "split", "cat_name"])


def save_label_manifest(path: Path, rows: list[dict]) -> None:
    """Append rows into labels manifest and keep latest entry per image name"""
    path.parent.mkdir(parents=True, exist_ok=True)
    old = load_label_manifest(path)
    new = pd.DataFrame(rows, columns=["image_name", "split", "cat_name"])

    merged = pd.concat([old, new], ignore_index=True)
    merged = merged.drop_duplicates(subset=["image_name"], keep="last")
    merged.to_csv(path, index=False)


def build_preview(img, i, stem, split, confidence):
    preview = cv2.resize(img, (640, 640), interpolation=cv2.INTER_LINEAR)
    info = f"{i + 1}/{len(selected_candidates)} id={stem} split={split} conf={confidence:.2f}"
    keys = " ".join(
        [
            f"0={CAT_NAMES[0]}",
            f"1={CAT_NAMES[1]}",
            f"2={CAT_NAMES[2]}",
            "q=quit",
        ]
    )
    cv2.putText(
        preview,
        info,
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 180, 255),
        2,
        cv2.LINE_AA,
    )
    cv2.putText(
        preview,
        keys,
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 220, 0),
        2,
        cv2.LINE_AA,
    )
    return preview


def suggest_params(trial, model_name: str) -> dict:
    if model_name == "RBF SVM":
        return {
            "clf__C": trial.suggest_float("clf__C", 1e-3, 3e1, log=True),
            "clf__gamma": trial.suggest_categorical(
                "clf__gamma",
                ["scale", "auto", 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
            ),
            "clf__class_weight": trial.suggest_categorical(
                "clf__class_weight", [None, "balanced"]
            ),
        }
    if model_name == "MLP":

        return {
            "clf__hidden_layer_sizes": trial.suggest_categorical(
                "clf__hidden_layer_sizes",
                [(64), (96), (128), (160), (192), (128, 64), (192, 96), (256, 128)],
            ),
            "clf__alpha": trial.suggest_float("clf__alpha", 1e-7, 3e-2, log=True),
        }
    if model_name == "Gradient Boosted Trees":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 80, 420),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 2e-1, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 5),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 7),
            "subsample": trial.suggest_categorical("subsample", [0.7, 0.85, 1.0]),
        }
    if model_name == "Random Forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 4, 20),
            "max_depth": trial.suggest_categorical("max_depth", [2, 3]),
            "ccp_alpha": trial.suggest_float("ccp_alpha", 0.01, 0.1),
        }
    raise ValueError(f"Unknown model name: {model_name}")

## Load and process files

In [ ]:
# Load already-exported ROI images and create TrackFrames for diversity filtering
existing_track_frames = []
for img_path in tqdm(list((EXPORT_ROOT / "images").rglob("*.jpg"))):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    existing_track_frames.append(
        tracking.TrackFrame(
            frame_hash=img_path.stem,
            image=img,
            bbox=tracking.utils.Bbox(xyxy=(0, 0, w - 1, h - 1)),
            object_name="na",
            confidence=-1,
        )
    )

In [ ]:
# Load videos, extract candidate frames, and detect objects into TrackFrames
candidates = []
prev_proc_frame = None

mock_inputs = [
    p
    for ext in ("*_raw.avi", "*_raw.mp4")
    for p in (PROJECT_ROOT / "datasets" / "mock_inputs").glob(ext)
]
raw_video = [
    p
    for ext in ("*_raw.avi", "*_raw.mp4")
    for p in (PROJECT_ROOT / "datasets" / "raw_video").glob(ext)
]
for vpath in tqdm(mock_inputs + raw_video):
    cap = cv2.VideoCapture(str(vpath))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % 2 == 0:
            proc_frame = processing.Frame(
                timestamp=datetime.now(),
                image=frame,
                prev_frame=prev_proc_frame,
                prev_track_mask=np.zeros(frame.shape[:2], dtype=np.uint8),
                forced_detection_run=True,
            )
            track_frames, _ = proc_frame.detect_objects()
            prev_proc_frame = proc_frame
            candidates.extend(track_frames)
        i += 1
    cap.release()

## Select and label frames

In [ ]:
# Select diverse and difficult TrackFrames for labeling
emb_matrix = np.stack(
    [tf.roi_embedding.astype(np.float32) for tf in candidates], axis=0
)
uncert = np.array([1.0 - tf.confidence for tf in candidates], dtype=np.float32)
existing_emb_matrix = (
    np.stack(
        [tf.roi_embedding.astype(np.float32) for tf in existing_track_frames],
        axis=0,
    )
    if len(existing_track_frames) > 0
    else None
)

selected = []
remaining = set(range(len(candidates)))
while len(selected) < TARGET_LABEL_FRAMES and remaining:
    best_idx = None
    best_score = -1.0
    for i in remaining:

        # calculate similarity to already-selected and existing embeddings
        reference_max_sim = 0.0
        if selected:
            selected_emb = emb_matrix[selected]
            reference_max_sim = max(
                reference_max_sim, np.max(selected_emb @ emb_matrix[i])
            )
        if existing_emb_matrix is not None:
            reference_max_sim = max(
                reference_max_sim, np.max(existing_emb_matrix @ emb_matrix[i])
            )

        # calculate score based on uncertainty and diversity
        score = uncert[i] * (1.0 - reference_max_sim)
        if score > best_score:
            best_score = score
            best_idx = i

    selected.append(best_idx)
    remaining.remove(best_idx)

selected_candidates = [candidates[i] for i in selected]

In [ ]:
# Manually label ROIs for classification labels
existing_manifest = load_label_manifest(LABEL_MANIFEST_PATH)
existing_ids = (
    existing_manifest["image_name"]
    .str.replace(".jpg", "", regex=False)
    .loc[lambda s: s.str.isdigit()]
    .astype(int)
    .tolist()
    if len(existing_manifest) > 0
    else []
)
next_id = max(existing_ids, default=0) + 1

indices = list(range(len(selected_candidates)))
random.shuffle(indices)
n_train = round(TRAIN_SPLIT * len(indices))
train_set = set(indices[:n_train])

review_records = []
for i, tf in enumerate(selected_candidates):
    split = SOURCE_SPLITS[0] if i in train_set else SOURCE_SPLITS[1]
    stem = f"{next_id + i:06d}"
    image_name = f"{stem}.jpg"
    roi = tf.roi.copy()

    cv2.imshow(
        "review",
        build_preview(
            roi,
            i,
            stem,
            split,
            tf.confidence,
        ),
    )
    key = cv2.waitKey(0) & 0xFF

    if key == ord("q"):
        break
    elif key == ord("0"):
        final_cat_name = CAT_NAMES[0]
    elif key == ord("1"):
        final_cat_name = CAT_NAMES[1]
    elif key == ord("2"):
        final_cat_name = CAT_NAMES[2]
    else:
        continue

    review_records.append(
        {
            "image_name": image_name,
            "split": split,
            "roi_bgr": roi,
            "cat_name": final_cat_name,
        }
    )

cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
# Export reviewed ROI crops and a single labels manifest file
manifest_rows = []
for rec in review_records:
    split = rec["split"]
    image_name = rec["image_name"]
    img_path = image_path_for_split(split, image_name)

    cv2.imwrite(str(img_path), rec["roi_bgr"])
    manifest_rows.append(
        {
            "image_name": image_name,
            "split": split,
            "cat_name": rec["cat_name"],
        }
    )

save_label_manifest(LABEL_MANIFEST_PATH, manifest_rows)

## Evaluate classifier models

In [ ]:
# Build train/val embedding matrices from manifest
manifest_df = load_label_manifest(LABEL_MANIFEST_PATH)
train_features, train_targets = [], []
val_features, val_targets = [], []

for row in tqdm(manifest_df.to_dict("records"), desc="Embedding manifest rows"):
    split = row["split"]
    image_name = row["image_name"]
    target = row["cat_name"]
    img = cv2.imread(str(image_path_for_split(split, image_name)))

    h, w = img.shape[:2]
    tf = tracking.TrackFrame(
        frame_hash=Path(image_name).stem,
        image=img,
        bbox=tracking.utils.Bbox(xyxy=(0, 0, w - 1, h - 1)),
        object_name="na",
        confidence=1,
    )
    emb = tf.roi_embedding.astype(np.float32)
    if split == SOURCE_SPLITS[0]:
        train_features.append(emb)
        train_targets.append(target)
    elif split == SOURCE_SPLITS[1]:
        val_features.append(emb)
        val_targets.append(target)

X_train = np.stack(train_features, axis=0)
y_train = np.array(train_targets)
X_val = np.stack(val_features, axis=0)
y_val = np.array(val_targets)

In [ ]:
# Instantiate model candidates, loading previously tuned models when present
MODEL_CANDIDATE_PATHS = {
    "RBF SVM": MODEL_CANDIDATES_DIR / "rbf_svm_tuned.joblib",
    "MLP": MODEL_CANDIDATES_DIR / "mlp_tuned.joblib",
    "Gradient Boosted Trees": MODEL_CANDIDATES_DIR
    / "gradient_boosted_trees_tuned.joblib",
    "Random Forest": MODEL_CANDIDATES_DIR / "random_forest_tuned.joblib",
}

default_model_specs = {
    "RBF SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    C=1.0,
                    gamma="scale",
                    class_weight="balanced",
                    probability=True,
                    random_state=42,
                ),
            ),
        ]
    ),
    "MLP": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                MLPClassifier(
                    hidden_layer_sizes=(128,),
                    activation="relu",
                    alpha=1e-4,
                    learning_rate_init=1e-3,
                    max_iter=400,
                    early_stopping=True,
                    random_state=42,
                ),
            ),
        ]
    ),
    "Gradient Boosted Trees": GradientBoostingClassifier(
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1,
    ),
}

model_specs = {}
for model_name, default_model in default_model_specs.items():
    model_path = MODEL_CANDIDATE_PATHS[model_name]
    if model_path.exists():
        model_specs[model_name] = joblib.load(model_path)
    else:
        model_specs[model_name] = default_model

In [ ]:
# Tune each model with Optuna TPE, then evaluate and compare on validation data
cat_counts = pd.Series(y_train).value_counts().to_numpy()
cv_splits = np.clip(int(cat_counts.min()), 2, 5)
cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

_bench_train = X_train[: min(32, len(X_train))]

tuned_models = {}
model_results = []
for model_name, model in model_specs.items():

    # define objective function
    def objective(trial):
        params = suggest_params(trial, model_name)
        estimator = clone(model)
        estimator.set_params(**params)
        scores = cross_val_score(
            estimator,
            X_train,
            y_train,
            scoring="f1_macro",
            cv=cv,
            n_jobs=1,
        )

        # Fit a separate instance to benchmark inference latency for the RPi5 constraint
        latency_est = clone(model)
        latency_est.set_params(**params)
        latency_est.fit(X_train, y_train)
        t0 = perf_counter()
        for _ in range(BENCHMARK_RUNS):
            latency_est.predict(_bench_train)
        ms_per_sample = (
            (perf_counter() - t0) / (BENCHMARK_RUNS * len(_bench_train)) * 1000
        )
        trial.set_user_attr("latency_ms", ms_per_sample * SCALE_FACTOR)

        return np.mean(scores)

    # optimize hyperparameters with latency constraint
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            constraints_func=lambda trial: (
                trial.user_attrs.get("latency_ms", float("inf")) - INFERENCE_BUDGET_MS,
            ),
            seed=42,
        ),
    )
    study.optimize(objective, n_trials=SEARCH_ITERS, show_progress_bar=False)

    # exclude model if no trial met the RPi5 latency budget
    feasible_trials = [
        t
        for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
        and t.user_attrs.get("latency_ms", float("inf")) <= INFERENCE_BUDGET_MS
    ]
    if not feasible_trials:
        print(f"No feasible trials within inference budget for {model_name} — skipping")
        continue

    best_feasible = max(feasible_trials, key=lambda t: t.value)

    # retrain best feasible model on full training set
    best_model = clone(model)
    best_model.set_params(**best_feasible.params)
    best_model.fit(X_train, y_train)

    # evaluate model inference time on validation set
    bench_samples = X_val[: min(len(X_val), 64)]
    t0 = perf_counter()
    for _ in range(BENCHMARK_RUNS):
        best_model.predict(bench_samples)
    ms_per_sample = (perf_counter() - t0) / (BENCHMARK_RUNS * len(bench_samples)) * 1000

    # store model metrics
    val_pred = best_model.predict(X_val)
    tuned_models[model_name] = best_model
    model_results.append(
        {
            "model": model_name,
            "cv_macro_f1": best_feasible.value,
            "accuracy": accuracy_score(y_val, val_pred),
            "macro_f1": f1_score(y_val, val_pred, average="macro"),
            "cv_latency_ms": ms_per_sample * SCALE_FACTOR,
            "latency_ms": best_feasible.user_attrs["latency_ms"],
            "best_params": best_feasible.params,
        }
    )

results_df = (
    pd.DataFrame(model_results)
    .sort_values(
        ["macro_f1", "accuracy", "latency_ms"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
display(results_df)

In [ ]:
# Export tuned candidates and best classifier

for model_name, model in tuned_models.items():
    joblib.dump(model, MODEL_CANDIDATE_PATHS[model_name])

best_row = results_df.iloc[0].to_dict()
best_model_name = best_row["model"]
best_model = tuned_models[best_model_name]

joblib.dump(best_model, MODEL_EXPORT_PATH)